In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tab_transformer_pytorch import TabTransformer

# ------------------------
# Data Loading and Preprocessing
# ------------------------

# Load data and drop unwanted columns
df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")
df = df.drop(['Gender', 'Weight'], axis=1)

# Define target column
target_col = "NObeyesdad"

# Identify categorical and numerical columns:
# Assume categorical columns are of object type
categorical_cols = [col for col in df.select_dtypes(include=['object']).columns.tolist() if col != target_col]
numerical_cols = df.select_dtypes(exclude=['object']).columns.tolist()
# Remove the target column from numerical columns (if present)
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

# Convert target to numeric labels
le_target = LabelEncoder()
df[target_col] = le_target.fit_transform(df[target_col])

# Process categorical columns: use label encoding so that each category is represented as an integer index.
for col in categorical_cols:
    le_cat = LabelEncoder()
    df[col] = le_cat.fit_transform(df[col])

# Scale numerical columns using z-score scaling
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Create separate DataFrames for categorical and continuous features
X_cat = df[categorical_cols]
X_cont = df[numerical_cols]
y = df[target_col]

# For the TabTransformer, we need the number of unique values per categorical feature.
categories = [int(X_cat[col].nunique()) for col in categorical_cols]
num_continuous = len(numerical_cols)
print("Categories (cardinalities):", categories)
print("Number of continuous features:", num_continuous)



Categorical columns: ['family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
Numerical columns: ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categories (cardinalities): [2, 2, 4, 2, 2, 4, 5]
Number of continuous features: 7


In [21]:
# ------------------------
# Train / Validation / Test Split
# ------------------------

# First, split into training+validation and test sets (80% train+val, 20% test)
X_cat_train_val, X_cat_test, X_cont_train_val, X_cont_test, y_train_val, y_test = train_test_split(
    X_cat, X_cont, y, test_size=0.2, stratify=y, random_state=42
)

# Then split training+validation into training and validation sets (80% train, 20% validation)
X_cat_train, X_cat_val, X_cont_train, X_cont_val, y_train, y_val = train_test_split(
    X_cat_train_val, X_cont_train_val, y_train_val, test_size=0.2, stratify=y_train_val, random_state=42
)

# ------------------------
# Convert DataFrames to torch tensors
# ------------------------

# For categorical data, convert to torch.long tensors
X_cat_train_tensor = torch.tensor(X_cat_train.values, dtype=torch.long)
X_cat_val_tensor   = torch.tensor(X_cat_val.values, dtype=torch.long)
X_cat_test_tensor  = torch.tensor(X_cat_test.values, dtype=torch.long)

# For continuous data, convert to torch.float32 tensors
X_cont_train_tensor = torch.tensor(X_cont_train.values, dtype=torch.float32)
X_cont_val_tensor   = torch.tensor(X_cont_val.values, dtype=torch.float32)
X_cont_test_tensor  = torch.tensor(X_cont_test.values, dtype=torch.float32)

# Target tensor
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

# ------------------------
# Create a Custom Dataset
# ------------------------

class TabularDataset(Dataset):
    def __init__(self, X_cat, X_cont, y):
        self.X_cat = X_cat
        self.X_cont = X_cont
        self.y = y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_cont[idx], self.y[idx]

train_dataset = TabularDataset(X_cat_train_tensor, X_cont_train_tensor, y_train_tensor)
val_dataset   = TabularDataset(X_cat_val_tensor, X_cont_val_tensor, y_val_tensor)
test_dataset  = TabularDataset(X_cat_test_tensor, X_cont_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SAINTTransformer(nn.Module):
    def __init__(self, categories, num_continuous, dim, depth, heads, attn_dropout, ff_dropout, mlp_hidden_mults, dim_out):
        """
        Args:
          categories: List of integers, each representing the number of unique categories for that feature.
          num_continuous: Number of continuous features.
          dim: Embedding dimension for categorical features and projection dimension.
          depth: Number of Transformer layers.
          heads: Number of attention heads.
          attn_dropout: Dropout probability in attention layers.
          ff_dropout: Dropout probability in feed-forward layers.
          mlp_hidden_mults: Tuple of multipliers for hidden layer sizes in the MLP head.
          dim_out: Output dimension (e.g. number of classes).
        """
        super(SAINTTransformer, self).__init__()
        
        # Embedding layers for categorical features.
        self.category_embeddings = nn.ModuleList([nn.Embedding(cat, dim) for cat in categories])
        
        # A simple MLP for processing continuous features.
        self.continuous_mlp = nn.Sequential(
            nn.Linear(num_continuous, dim),
            nn.ReLU()
        )
        
        # Combine all features (categorical + continuous) by treating each as a token.
        # Total number of tokens = len(categories) + 1
        self.num_tokens = len(categories) + 1
        
        # Transformer encoder: using PyTorch's built-in TransformerEncoder.
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dropout=attn_dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        
        # MLP Head: input dimension is (num_tokens * dim) since we flatten the transformer output.
        mlp_input_dim = self.num_tokens * dim
        mlp_layers = []
        hidden_dims = [mlp_input_dim * mult for mult in mlp_hidden_mults]
        prev_dim = mlp_input_dim
        for h_dim in hidden_dims:
            mlp_layers.append(nn.Linear(prev_dim, h_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(ff_dropout))
            prev_dim = h_dim
        mlp_layers.append(nn.Linear(prev_dim, dim_out))
        self.mlp = nn.Sequential(*mlp_layers)
    
    def forward(self, X_cat, X_cont):
        """
        Args:
          X_cat: Tensor of shape (batch, num_categorical_features)
          X_cont: Tensor of shape (batch, num_continuous_features)
        Returns:
          Output logits of shape (batch, dim_out)
        """
        # Process each categorical feature with its embedding.
        cat_embeds = [embed(X_cat[:, i]) for i, embed in enumerate(self.category_embeddings)]
        # Each embedding has shape (batch, dim). Stack along token dimension.
        cat_tokens = torch.stack(cat_embeds, dim=1)  # shape: (batch, num_categories, dim)
        
        # Process continuous features.
        cont_embed = self.continuous_mlp(X_cont)  # shape: (batch, dim)
        # Expand to create a token: shape (batch, 1, dim)
        cont_token = cont_embed.unsqueeze(1)
        
        # Concatenate categorical and continuous tokens.
        tokens = torch.cat([cat_tokens, cont_token], dim=1)  # shape: (batch, num_tokens, dim)
        
        # Permute tokens to shape (num_tokens, batch, dim) for the transformer.
        tokens = tokens.permute(1, 0, 2)
        
        # Apply the transformer encoder.
        transformer_out = self.transformer_encoder(tokens)  # shape: (num_tokens, batch, dim)
        
        # Permute back and flatten tokens.
        transformer_out = transformer_out.permute(1, 0, 2).contiguous()  # (batch, num_tokens, dim)
        transformer_flat = transformer_out.view(transformer_out.size(0), -1)  # (batch, num_tokens*dim)
        
        # Final output via the MLP head.
        logits = self.mlp(transformer_flat)  # (batch, dim_out)
        return logits

# Example usage:
# Define your dataset parameters:


# Instantiate the SAINT Transformer with chosen hyperparameters.
model = SAINTTransformer(
    categories=tuple(categories),
    num_continuous=num_continuous,
    dim=64,                # embedding/projection dimension
    depth=3,               # number of Transformer layers
    heads=4,               # attention heads per layer
    attn_dropout=0.2,      # dropout in attention
    ff_dropout=0.2,        # dropout in feed-forward network
    mlp_hidden_mults=(4,2),# hidden layer multipliers for the MLP head
    dim_out=7              # number of classes for output
)

print(model)
device = torch.device("mps")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0004815730708625121, weight_decay = 5.127077152810453e-07)

SAINTTransformer(
  (category_embeddings): ModuleList(
    (0-1): 2 x Embedding(2, 64)
    (2): Embedding(4, 64)
    (3-4): 2 x Embedding(2, 64)
    (5): Embedding(4, 64)
    (6): Embedding(5, 64)
  )
  (continuous_mlp): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=

/Users/alexkuai/PythonStuff/miniconda3/envs/pytorch/lib/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [22]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Define your device and assume train_loader and val_loader are predefined DataLoader objects.
device = torch.device("mps" if torch.mps.is_available() else "cpu")

# Assume categories and num_continuous are defined based on your dataset.
# For example:
# categories = [cardinality1, cardinality2, ...]
# num_continuous = number_of_continuous_features

def objective(trial):
    # Define candidate dimensions and possible heads.
    possible_dims = [64, 128, 256]
    # Only allow heads that divide all candidate dimensions.
    possible_heads = [h for h in [2, 4, 8, 16, 32] if all(d % h == 0 for d in possible_dims)]
    
    # Sample the number of attention heads from the valid set.
    heads = trial.suggest_categorical("heads", possible_heads)
    # Sample the embedding dimension from the candidate set.
    dim = trial.suggest_categorical("dim", possible_dims)
    # Since every candidate in possible_dims is divisible by any h in possible_heads, no further filtering is required.
    
    depth = trial.suggest_int("depth", 2, 5)
    attn_dropout = trial.suggest_float("attn_dropout", 0.1, 0.5)
    ff_dropout = trial.suggest_float("ff_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-5, log=True)
    num_epochs = trial.suggest_int("num_epochs", 10, 50)
    mlp_multiplier1 = trial.suggest_int("mlp_multiplier1", 2, 6)
    mlp_multiplier2 = trial.suggest_int("mlp_multiplier2", 1, 4)
    mlp_hidden_mults = (mlp_multiplier1, mlp_multiplier2)
    # Instantiate your FTTransformer model with the suggested hyperparameters.
    model = SAINTTransformer(
        categories=tuple(categories),
        num_continuous=num_continuous,
        dim=dim,
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        mlp_hidden_mults=mlp_hidden_mults,
        dim_out=7  # Assuming 7 classes
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_val_acc = 0.0
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for X_cat_batch, X_cont_batch, y_batch in train_loader:
            X_cat_batch = X_cat_batch.to(device)
            X_cont_batch = X_cont_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_cat_batch, X_cont_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * y_batch.size(0)
        avg_train_loss = train_loss / len(train_loader.dataset)

        # Evaluate on validation set.
        model.eval()
        val_loss = 0.0
        val_true = []
        val_pred = []
        with torch.no_grad():
            for X_cat_batch, X_cont_batch, y_batch in val_loader:
                X_cat_batch = X_cat_batch.to(device)
                X_cont_batch = X_cont_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model(X_cat_batch, X_cont_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * y_batch.size(0)
                preds = outputs.argmax(dim=1)
                val_true.extend(y_batch.cpu().numpy())
                val_pred.extend(preds.cpu().numpy())
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = np.mean(np.array(val_true) == np.array(val_pred))

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        best_val_loss = min(best_val_loss, avg_val_loss)

    trial.set_user_attr("val_accuracy", best_val_acc)
    return best_val_loss

# Create and optimize the study.
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100)

print("Best trial:")
trial = study.best_trial
print("  Best Validation Loss: {:.4f}".format(trial.value))
print("  Best Hyperparameters: ", trial.params)
print("  Best Validation Accuracy: {:.4f}".format(trial.user_attrs["val_accuracy"]))


[I 2025-02-26 17:08:11,708] A new study created in memory with name: no-name-80287928-efab-4d51-a230-4f9a2ae5abb6
/Users/alexkuai/PythonStuff/miniconda3/envs/pytorch/lib/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
[I 2025-02-26 17:09:03,744] Trial 0 finished with value: 0.6995424525977592 and parameters: {'heads': 8, 'dim': 256, 'depth': 2, 'attn_dropout': 0.23908334086441868, 'ff_dropout': 0.40316607838321583, 'learning_rate': 3.737447791462216e-05, 'weight_decay': 1.1300908179246337e-06, 'num_epochs': 45, 'mlp_multiplier1': 2, 'mlp_multiplier2': 4}. Best is trial 0 with value: 0.6995424525977592.
[I 2025-02-26 17:09:49,591] Trial 1 finished with value: 0.8679503371729653 and parameters: {'heads': 4, 'dim': 128, 'depth': 4, 'attn_dropout': 0.4919255286701038, 'ff_drop

[I 2025-02-26 17:20:41,168] Trial 79 pruned. 
[I 2025-02-26 17:20:43,393] Trial 80 pruned. 
[I 2025-02-26 17:20:44,791] Trial 81 pruned. 
[I 2025-02-26 17:20:46,079] Trial 82 pruned. 
[I 2025-02-26 17:20:46,786] Trial 83 pruned. 
[I 2025-02-26 17:20:48,170] Trial 84 pruned. 
[I 2025-02-26 17:20:49,259] Trial 85 pruned. 
[I 2025-02-26 17:20:53,348] Trial 86 pruned. 
[I 2025-02-26 17:20:55,050] Trial 87 pruned. 
[I 2025-02-26 17:20:58,967] Trial 88 pruned. 
[I 2025-02-26 17:21:00,152] Trial 89 pruned. 
[I 2025-02-26 17:21:01,544] Trial 90 pruned. 
[I 2025-02-26 17:21:02,579] Trial 91 pruned. 
[I 2025-02-26 17:21:03,515] Trial 92 pruned. 
[I 2025-02-26 17:21:04,514] Trial 93 pruned. 
[I 2025-02-26 17:21:05,440] Trial 94 pruned. 
[I 2025-02-26 17:21:08,824] Trial 95 pruned. 
[I 2025-02-26 17:21:12,113] Trial 96 pruned. 
[I 2025-02-26 17:21:13,542] Trial 97 pruned. 
[I 2025-02-26 17:21:16,917] Trial 98 pruned. 
[I 2025-02-26 17:21:20,606] Trial 99 pruned. 


Best trial:
  Best Validation Loss: 0.6936
  Best Hyperparameters:  {'heads': 16, 'dim': 64, 'depth': 4, 'attn_dropout': 0.14646928352942898, 'ff_dropout': 0.32977370067437123, 'learning_rate': 0.00014030387313824908, 'weight_decay': 5.802937611551732e-06, 'num_epochs': 26, 'mlp_multiplier1': 4, 'mlp_multiplier2': 4}
  Best Validation Accuracy: 0.8225


In [27]:
model = SAINTTransformer(
    categories=tuple(categories),
    num_continuous=num_continuous,
    dim=128,                # embedding/projection dimension
    depth=4,               # number of Transformer layers
    heads=16,               # attention heads per layer
    attn_dropout=0.14646928352942898,      # dropout in attention
    ff_dropout=0.32977370067437123,        # dropout in feed-forward network
    mlp_hidden_mults=(4,4),# hidden layer multipliers for the MLP head
    dim_out=7              # number of classes for output
)

print(model)
device = torch.device("mps")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00014030387313824908, weight_decay = 5.802937611551732e-06)

SAINTTransformer(
  (category_embeddings): ModuleList(
    (0-1): 2 x Embedding(2, 128)
    (2): Embedding(4, 128)
    (3-4): 2 x Embedding(2, 128)
    (5): Embedding(4, 128)
    (6): Embedding(5, 128)
  )
  (continuous_mlp): Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): ReLU()
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.14646928352942898, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.14646928352942898, inplace=False)

/Users/alexkuai/PythonStuff/miniconda3/envs/pytorch/lib/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [28]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score
import numpy as np

num_epochs = 26
best_val_loss = float('inf')

for epoch in range(num_epochs):
    start_epoch = time.time()
    
    # ---- Training Phase ----
    model.train()
    total_loss = 0.0
    train_true = []
    train_pred = []
    train_prob_list = []  # to accumulate probability vectors for each sample
    
    for X_cat_batch, X_cont_batch, y_batch in train_loader:
        X_cat_batch = X_cat_batch.to(device)
        X_cont_batch = X_cont_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_cat_batch, X_cont_batch)  # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y_batch.size(0)
        
        preds = outputs.argmax(dim=1)
        train_true.extend(y_batch.cpu().numpy())
        train_pred.extend(preds.cpu().numpy())
        # For multiclass, store the full probability vector
        probs = torch.softmax(outputs, dim=1)
        train_prob_list.append(probs.cpu().detach().numpy())
    
    avg_train_loss = total_loss / len(train_loader.dataset)
    train_acc = accuracy_score(train_true, train_pred)
    train_precision = precision_score(train_true, train_pred, average='macro', zero_division=0)
    train_recall = recall_score(train_true, train_pred, average='macro', zero_division=0)
    
    # Concatenate probability vectors over all batches
    train_prob = np.concatenate(train_prob_list, axis=0)
    try:
        train_auc = roc_auc_score(train_true, train_prob, multi_class='ovr', average='macro')
    except ValueError:
        train_auc = np.nan
    
    # ---- Validation Phase ----
    model.eval()
    total_val_loss = 0.0
    val_true = []
    val_pred = []
    val_prob_list = []
    with torch.no_grad():
        for X_cat_batch, X_cont_batch, y_batch in val_loader:
            X_cat_batch = X_cat_batch.to(device)
            X_cont_batch = X_cont_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(X_cat_batch, X_cont_batch)
            loss = criterion(outputs, y_batch)
            total_val_loss += loss.item() * y_batch.size(0)
            
            preds = outputs.argmax(dim=1)
            val_true.extend(y_batch.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())
            probs = torch.softmax(outputs, dim=1)
            val_prob_list.append(probs.cpu().detach().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    val_acc = accuracy_score(val_true, val_pred)
    val_precision = precision_score(val_true, val_pred, average='macro', zero_division=0)
    val_recall = recall_score(val_true, val_pred, average='macro', zero_division=0)
    val_prob = np.concatenate(val_prob_list, axis=0)
    try:
        val_auc = roc_auc_score(val_true, val_prob, multi_class='ovr', average='macro')
    except ValueError:
        val_auc = np.nan

    end_epoch = time.time()
    epoch_duration = end_epoch - start_epoch

    print(f"Epoch {epoch+1}/{num_epochs} - Duration: {epoch_duration:.2f}s")
    print(f"  Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.4f}, AUC: {train_auc:.4f}, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}")
    print(f"  Val   Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}, AUC: {val_auc:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}")
    
    # Save the best model based on validation loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "SAINT_valbest.pt")
        print("Best model updated and saved.")

print("Training complete.")


Epoch 1/26 - Duration: 2.01s
  Train Loss: 1.4204, Acc: 0.4615, AUC: 0.8017, Precision: 0.4378, Recall: 0.4527
  Val   Loss: 1.1107, Acc: 0.6243, AUC: 0.8889, Precision: 0.6211, Recall: 0.6172
Best model updated and saved.
Epoch 2/26 - Duration: 1.37s
  Train Loss: 0.9990, Acc: 0.6422, AUC: 0.8971, Precision: 0.6297, Recall: 0.6367
  Val   Loss: 0.9205, Acc: 0.6479, AUC: 0.9194, Precision: 0.6603, Recall: 0.6367
Best model updated and saved.
Epoch 3/26 - Duration: 1.30s
  Train Loss: 0.8508, Acc: 0.6985, AUC: 0.9215, Precision: 0.6870, Recall: 0.6942
  Val   Loss: 0.9627, Acc: 0.6982, AUC: 0.9198, Precision: 0.7117, Recall: 0.6950
Epoch 4/26 - Duration: 1.42s
  Train Loss: 0.7918, Acc: 0.7059, AUC: 0.9329, Precision: 0.7004, Recall: 0.7026
  Val   Loss: 0.8178, Acc: 0.7337, AUC: 0.9393, Precision: 0.7465, Recall: 0.7280
Best model updated and saved.
Epoch 5/26 - Duration: 1.28s
  Train Loss: 0.7012, Acc: 0.7489, AUC: 0.9454, Precision: 0.7419, Recall: 0.7445
  Val   Loss: 0.8369, Acc: 

In [29]:
# ------------------------
# Evaluate on Test Set
# ------------------------
model.eval()
test_true = []
test_pred = []
test_prob_list = []
with torch.no_grad():
    for X_cat_batch, X_cont_batch, y_batch in test_loader:
        X_cat_batch = X_cat_batch.to(device)
        X_cont_batch = X_cont_batch.to(device)
        y_batch = y_batch.to(device)
        outputs = model(X_cat_batch, X_cont_batch)
        preds = outputs.argmax(dim=1)
        test_true.extend(y_batch.cpu().numpy())
        test_pred.extend(preds.cpu().numpy())
        probs = torch.softmax(outputs, dim=1)
        test_prob_list.append(probs.cpu().detach().numpy())
        
test_acc = accuracy_score(test_true, test_pred)
test_precision = precision_score(test_true, test_pred, average='macro', zero_division=0)
test_recall = recall_score(test_true, test_pred, average='macro', zero_division=0)
test_prob = np.concatenate(test_prob_list, axis=0)
try:
    test_auc = roc_auc_score(test_true, test_prob, multi_class='ovr', average='macro')
except ValueError:
    test_auc = np.nan
print("Test Metrics:")
print(f"  Acc: {test_acc:.4f}, AUC: {test_auc:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}")


Test Metrics:
  Acc: 0.7825, AUC: 0.9451, Precision: 0.7775, Recall: 0.7767
